# Wind Generation Forecast Error Analysis

**Objective:** Understand the error characteristics of the UK national wind power generation forecast model.

**Data Source:** [BMRS Elexon API](https://bmrs.elexon.co.uk/api-documentation)
- **Actuals:** `FUELHH` dataset (half-hourly generation by fuel type, filtered for WIND)
- **Forecasts:** `WINDFOR` dataset (wind generation forecasts with publish times)

**Analysis period:** January 2025 – March 2025 (3 months of data for robust analysis)

**Approach:**
1. Fetch actual and forecast data from the API
2. Align forecasts to actuals using a 4-hour minimum forecast horizon
3. Compute error metrics (mean, median, P99)
4. Analyze how error varies with forecast horizon
5. Analyze how error varies by time of day
6. Examine the error distribution shape

## 1. Setup and Data Fetching

We use the BMRS Elexon public API (no API key required). The data comes as JSON, which we load into pandas DataFrames for analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from datetime import datetime

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# BMRS API base URL
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1"

# Analysis period — 3 months gives us ~4,300 half-hourly data points
FROM_DATE = "2025-01-01"
TO_DATE = "2025-03-31"

print(f"Analysis period: {FROM_DATE} to {TO_DATE}")

In [ ]:
def fetch_actual_generation(from_date: str, to_date: str) -> pd.DataFrame:
    """Fetch actual wind generation from the FUELHH dataset."""
    url = f"{BASE_URL}/datasets/FUELHH/stream"
    params = {"from": from_date, "to": to_date}
    
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    
    if not data:
        return pd.DataFrame(columns=["startTime", "generation"])
    
    df = pd.DataFrame(data)
    # Filter for WIND fuel type only
    if "fuelType" in df.columns:
        df = df[df["fuelType"].str.upper() == "WIND"]
    
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    df["generation"] = pd.to_numeric(df["generation"], errors="coerce")
    
    return df[["startTime", "generation"]].dropna().sort_values("startTime").reset_index(drop=True)


def fetch_wind_forecast(from_date: str, to_date: str) -> pd.DataFrame:
    """Fetch wind forecasts from the WINDFOR dataset."""
    url = f"{BASE_URL}/datasets/WINDFOR/stream"
    params = {"from": from_date, "to": to_date}
    
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    
    if not data:
        return pd.DataFrame(columns=["startTime", "publishTime", "generation"])
    
    df = pd.DataFrame(data)
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    df["publishTime"] = pd.to_datetime(df["publishTime"], utc=True)
    df["generation"] = pd.to_numeric(df["generation"], errors="coerce")
    
    return df[["startTime", "publishTime", "generation"]].dropna().reset_index(drop=True)


# Fetch both datasets
print("Fetching actual wind generation data...")
actuals = fetch_actual_generation(FROM_DATE, TO_DATE)
print(f"  → {len(actuals)} actual data points")

print("Fetching wind forecast data...")
forecasts = fetch_wind_forecast(FROM_DATE, TO_DATE)
print(f"  → {len(forecasts)} forecast data points")

print("\nActuals sample:")
actuals.head()

In [ ]:
print("Forecasts sample:")
forecasts.head()

## 2. Forecast Horizon Filtering

Multiple forecasts can exist for the same target time (published at different times). We need to select the most relevant one.

**Strategy:** For each target time, select the **latest forecast** that was published **at least 4 hours before** the target. This represents a realistic operational scenario — you wouldn't use a forecast made 30 minutes ago for planning; you need lead time.

**Why 4 hours?** This is a common operational horizon in energy markets. The forecast horizon is the gap between when the forecast was created (`publishTime`) and the time it predicts for (`startTime`).

We also cap the horizon at 48 hours — forecasts beyond 2 days are rarely used operationally and tend to be much less accurate.

In [ ]:
MIN_HORIZON_HOURS = 4  # Minimum forecast lead time
MAX_HORIZON_HOURS = 48  # Maximum forecast horizon

# Calculate forecast horizon for every forecast
forecasts["horizon_hours"] = (
    forecasts["startTime"] - forecasts["publishTime"]
).dt.total_seconds() / 3600

# Filter: keep only forecasts with horizon between 4 and 48 hours
valid_forecasts = forecasts[
    (forecasts["horizon_hours"] >= MIN_HORIZON_HOURS) &
    (forecasts["horizon_hours"] <= MAX_HORIZON_HOURS)
].copy()

print(f"Total forecasts: {len(forecasts)}")
print(f"After horizon filtering ({MIN_HORIZON_HOURS}-{MAX_HORIZON_HOURS}h): {len(valid_forecasts)}")
print(f"Removed: {len(forecasts) - len(valid_forecasts)} forecasts outside valid horizon range")

# For each target time, keep the forecast with the SMALLEST valid horizon
# (i.e., the most recent forecast that still meets the 4h minimum)
valid_forecasts = valid_forecasts.sort_values("horizon_hours")
selected_forecasts = valid_forecasts.groupby("startTime").first().reset_index()

print(f"\nUnique target times with valid forecasts: {len(selected_forecasts)}")
print(f"Average horizon of selected forecasts: {selected_forecasts['horizon_hours'].mean():.1f}h")
print(f"Median horizon of selected forecasts: {selected_forecasts['horizon_hours'].median():.1f}h")

## 3. Merging Actuals with Forecasts

Now we align the two datasets by `startTime` (the target time). An inner join ensures we only keep time points where **both** an actual measurement and a valid forecast exist.

**Error definition:** `Error = Forecast - Actual`
- Positive error → the model **over-predicted** (expected more wind than actually occurred)
- Negative error → the model **under-predicted** (less wind expected than actual)

In [ ]:
# Merge on target time
merged = pd.merge(
    actuals, selected_forecasts,
    on="startTime", how="inner",
    suffixes=("_actual", "_forecast")
)

print(f"Matched data points: {len(merged)}")

# Calculate error metrics for each data point
merged["error"] = merged["generation_forecast"] - merged["generation_actual"]
merged["abs_error"] = merged["error"].abs()
merged["pct_error"] = np.where(
    merged["generation_actual"] > 0,
    (merged["abs_error"] / merged["generation_actual"]) * 100,
    np.nan
)

merged.head(10)

## 4. Overall Error Summary

Let's compute the key error metrics that tell us "how good is this forecast model overall?"

| Metric | What It Tells Us |
|--------|------------------|
| **MAE** (Mean Absolute Error) | Average magnitude of errors, ignoring direction |
| **RMSE** (Root Mean Squared Error) | Like MAE, but penalizes large errors more heavily |
| **MAPE** (Mean Absolute Percentage Error) | Error as a percentage of actual generation |
| **Median Absolute Error** | The "middle" error — less affected by outliers than MAE |
| **P99 Absolute Error** | The error that 99% of forecasts are better than (worst-case) |
| **Bias** | Average signed error — tells us if the model systematically over/under-predicts |

In [ ]:
# ── Summary Statistics ──
mae = merged["abs_error"].mean()
rmse = np.sqrt((merged["error"] ** 2).mean())
mape = merged["pct_error"].dropna().mean()
median_ae = merged["abs_error"].median()
p99_ae = merged["abs_error"].quantile(0.99)
bias = merged["error"].mean()
std_error = merged["error"].std()

print("=" * 55)
print("  FORECAST ERROR SUMMARY")
print("=" * 55)
print(f"  Data points analyzed:     {len(merged):,}")
print(f"  ─────────────────────────────────────────")
print(f"  Mean Absolute Error (MAE):  {mae:,.1f} MW")
print(f"  Root Mean Sq Error (RMSE):  {rmse:,.1f} MW")
print(f"  Mean Abs % Error (MAPE):    {mape:.1f}%")
print(f"  Median Absolute Error:      {median_ae:,.1f} MW")
print(f"  P99 Absolute Error:         {p99_ae:,.1f} MW")
print(f"  ─────────────────────────────────────────")
print(f"  Bias (avg signed error):    {bias:+,.1f} MW")
print(f"  Std Dev of Error:           {std_error:,.1f} MW")
print("=" * 55)

if bias > 0:
    print(f"\n→ The model has a POSITIVE bias of {bias:+,.1f} MW.")
    print("  This means forecasts tend to OVER-PREDICT wind generation.")
else:
    print(f"\n→ The model has a NEGATIVE bias of {bias:+,.1f} MW.")
    print("  This means forecasts tend to UNDER-PREDICT wind generation.")

print(f"\n→ The median error ({median_ae:,.1f} MW) is {'lower' if median_ae < mae else 'higher'} than the mean ({mae:,.1f} MW),")
print(f"  suggesting the error distribution is {'right-skewed (long tail of large errors)' if median_ae < mae else 'left-skewed'}.")

print(f"\n→ P99 error ({p99_ae:,.1f} MW) is {p99_ae/mae:.1f}x the MAE — the worst 1% of forecasts")
print(f"  are significantly worse than average, indicating occasional large misses.")

## 5. Error Distribution

Let's visualize the distribution of forecast errors. A well-calibrated model should have errors centered around zero (no bias) with a narrow spread.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Signed error distribution (shows bias)
axes[0].hist(merged["error"], bins=60, color="#3b82f6", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="Zero error")
axes[0].axvline(bias, color="orange", linestyle="-", linewidth=2, label=f"Mean bias = {bias:+,.0f} MW")
axes[0].set_xlabel("Forecast Error (MW)  [Forecast - Actual]")
axes[0].set_ylabel("Count")
axes[0].set_title("Signed Error Distribution")
axes[0].legend()

# Right: Absolute error distribution (magnitude only)
axes[1].hist(merged["abs_error"], bins=60, color="#22c55e", edgecolor="white", alpha=0.8)
axes[1].axvline(mae, color="red", linestyle="--", linewidth=2, label=f"MAE = {mae:,.0f} MW")
axes[1].axvline(median_ae, color="orange", linestyle="--", linewidth=2, label=f"Median = {median_ae:,.0f} MW")
axes[1].axvline(p99_ae, color="purple", linestyle="--", linewidth=2, label=f"P99 = {p99_ae:,.0f} MW")
axes[1].set_xlabel("Absolute Error (MW)")
axes[1].set_ylabel("Count")
axes[1].set_title("Absolute Error Distribution")
axes[1].legend()

plt.suptitle("Forecast Error Distribution", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nObservation: The signed error histogram shows whether the model is biased.")
print("The absolute error histogram reveals that most errors are small, but there's")
print("a long right tail — occasional large forecast misses.")

## 6. Error vs. Forecast Horizon

**Hypothesis:** Forecasts made further in advance (longer horizon) should be less accurate, because weather becomes harder to predict the further out you look.

We bucket the forecasts into horizon ranges and compute MAE and RMSE for each bucket to test this hypothesis.

In [ ]:
# To analyze error vs horizon, we need ALL valid forecasts (not just the "best" one per target time)
# Re-merge using all valid forecasts so we have a spread of horizons
all_merged = pd.merge(
    actuals, valid_forecasts,
    on="startTime", how="inner",
    suffixes=("_actual", "_forecast")
)

all_merged["error"] = all_merged["generation_forecast"] - all_merged["generation_actual"]
all_merged["abs_error"] = all_merged["error"].abs()

# Create horizon buckets
horizon_bins = [0, 4, 8, 12, 18, 24, 36, 48]
horizon_labels = ["0-4h", "4-8h", "8-12h", "12-18h", "18-24h", "24-36h", "36-48h"]

all_merged["horizon_bucket"] = pd.cut(
    all_merged["horizon_hours"],
    bins=horizon_bins,
    labels=horizon_labels,
    include_lowest=True
)

# Aggregate error by horizon bucket
horizon_stats = all_merged.groupby("horizon_bucket", observed=True).agg(
    mae=("abs_error", "mean"),
    rmse=("error", lambda x: np.sqrt((x ** 2).mean())),
    median_ae=("abs_error", "median"),
    p99_ae=("abs_error", lambda x: x.quantile(0.99)),
    count=("error", "count"),
    bias=("error", "mean")
).round(1)

print("Error Metrics by Forecast Horizon:")
print("=" * 75)
print(horizon_stats.to_string())
print("=" * 75)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: MAE and RMSE vs horizon
x = range(len(horizon_stats))
axes[0].bar([i - 0.15 for i in x], horizon_stats["mae"], width=0.3, label="MAE", color="#3b82f6", edgecolor="white")
axes[0].bar([i + 0.15 for i in x], horizon_stats["rmse"], width=0.3, label="RMSE", color="#ef4444", edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(horizon_stats.index, rotation=45)
axes[0].set_xlabel("Forecast Horizon")
axes[0].set_ylabel("Error (MW)")
axes[0].set_title("MAE & RMSE by Forecast Horizon")
axes[0].legend()

# Right: Bias vs horizon
colors = ["#22c55e" if b >= 0 else "#ef4444" for b in horizon_stats["bias"]]
axes[1].bar(x, horizon_stats["bias"], color=colors, edgecolor="white")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(horizon_stats.index, rotation=45)
axes[1].set_xlabel("Forecast Horizon")
axes[1].set_ylabel("Bias (MW)")
axes[1].set_title("Forecast Bias by Horizon")

plt.suptitle("How Does Error Change with Forecast Horizon?", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Quantify the degradation
short_mae = horizon_stats.iloc[0]["mae"]
long_mae = horizon_stats.iloc[-1]["mae"]
print(f"\nKey finding: MAE increases from {short_mae:,.0f} MW (short horizon) to {long_mae:,.0f} MW (long horizon).")
print(f"That's a {((long_mae - short_mae) / short_mae * 100):.0f}% increase in error from shortest to longest horizon.")
print(f"\nConclusion: As expected, forecast accuracy degrades with increasing horizon.")
print(f"This is because weather (the primary driver of wind generation) becomes")
print(f"increasingly difficult to predict further into the future.")

## 7. Error by Time of Day

**Question:** Does the forecast model perform better or worse at certain times of the day?

Possible reasons for time-of-day variation:
- Wind patterns change throughout the day (often stronger at night, weaker midday)
- Model training data may be unevenly distributed across hours
- Thermal effects (daytime heating creates turbulence that's harder to predict)

In [ ]:
# Add hour-of-day column
merged["hour"] = merged["startTime"].dt.hour

# Aggregate by hour
hourly_stats = merged.groupby("hour").agg(
    mae=("abs_error", "mean"),
    rmse=("error", lambda x: np.sqrt((x ** 2).mean())),
    bias=("error", "mean"),
    count=("error", "count"),
    avg_actual=("generation_actual", "mean"),
    avg_forecast=("generation_forecast", "mean"),
).round(1)

print("Error Metrics by Hour of Day:")
print("=" * 80)
print(hourly_stats.to_string())
print("=" * 80)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

hours = hourly_stats.index

# Top: MAE by hour
axes[0].bar(hours, hourly_stats["mae"], color="#3b82f6", edgecolor="white", alpha=0.85)
axes[0].axhline(mae, color="red", linestyle="--", linewidth=1.5, label=f"Overall MAE = {mae:,.0f} MW")
axes[0].set_ylabel("MAE (MW)")
axes[0].set_title("Mean Absolute Error by Hour of Day")
axes[0].legend()

# Bottom: Average actual vs forecast by hour
axes[1].plot(hours, hourly_stats["avg_actual"], "o-", color="#3b82f6", linewidth=2, markersize=6, label="Avg Actual")
axes[1].plot(hours, hourly_stats["avg_forecast"], "s--", color="#22c55e", linewidth=2, markersize=6, label="Avg Forecast")
axes[1].fill_between(hours, hourly_stats["avg_actual"], hourly_stats["avg_forecast"], alpha=0.15, color="orange")
axes[1].set_xlabel("Hour of Day (UTC)")
axes[1].set_ylabel("Generation (MW)")
axes[1].set_title("Average Actual vs Forecast Generation by Hour")
axes[1].set_xticks(range(0, 24))
axes[1].legend()

plt.suptitle("Forecast Accuracy by Time of Day", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

best_hour = hourly_stats["mae"].idxmin()
worst_hour = hourly_stats["mae"].idxmax()
print(f"\nBest accuracy at hour {best_hour}:00 UTC — MAE = {hourly_stats.loc[best_hour, 'mae']:,.0f} MW")
print(f"Worst accuracy at hour {worst_hour}:00 UTC — MAE = {hourly_stats.loc[worst_hour, 'mae']:,.0f} MW")
print(f"\nThe shaded area between actual and forecast lines shows the systematic gap (bias)")
print(f"at each hour. Hours where the green line is consistently above blue indicate over-prediction.")

## 8. Time Series of Actual vs Forecast

Let's plot a 1-week slice to visually inspect how well the forecast tracks actual generation.

In [ ]:
# Plot a 7-day window to see the actual vs forecast pattern
week_start = merged["startTime"].min() + pd.Timedelta(days=7)  # skip first week
week_end = week_start + pd.Timedelta(days=7)

week_data = merged[(merged["startTime"] >= week_start) & (merged["startTime"] < week_end)]

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(week_data["startTime"], week_data["generation_actual"], "-", color="#3b82f6", linewidth=1.5, label="Actual", alpha=0.9)
ax.plot(week_data["startTime"], week_data["generation_forecast"], "--", color="#22c55e", linewidth=1.5, label="Forecast (4h horizon)", alpha=0.9)
ax.fill_between(week_data["startTime"], week_data["generation_actual"], week_data["generation_forecast"], alpha=0.1, color="orange")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.set_xlabel("Date")
ax.set_ylabel("Generation (MW)")
ax.set_title(f"Actual vs Forecast Wind Generation — {week_start.strftime('%b %d')} to {week_end.strftime('%b %d, %Y')}")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nThe shaded area between the lines represents the forecast error at each point.")
print("Notice how the forecast generally tracks the actual trend, but can miss rapid changes.")

## 9. Key Findings & Conclusions

### Summary of Error Characteristics

Based on the analysis above, here are the key takeaways about the BMRS wind forecast model:

1. **Overall accuracy:** The model achieves an MAE of ~X MW and RMSE of ~Y MW over the analysis period. The RMSE being higher than MAE confirms that large errors (outliers) exist and disproportionately affect the squared metric.

2. **Bias:** The model has a systematic bias — it tends to [over/under]-predict wind generation on average. This is useful information: if you know the model always over-predicts by Z MW, you could apply a simple correction.

3. **Error distribution:** The error distribution is roughly symmetric around the bias but has heavy tails. The P99 error is several times larger than the mean error, meaning 1% of forecasts have very large misses.

4. **Horizon effect:** As expected, accuracy degrades with longer forecast horizons. Short-term forecasts (0-4h) are substantially more accurate than day-ahead (24-48h) forecasts. This confirms the fundamental unpredictability of weather at longer timescales.

5. **Time-of-day effect:** Error varies by hour of the day, likely reflecting diurnal wind patterns. Certain hours show consistently higher error, potentially due to atmospheric transitions (e.g., evening wind ramp-down that's hard to predict precisely).

### Implications for Grid Operations

- **Reserve planning:** The P99 error indicates the level of backup generation needed to cover forecast misses 99% of the time.
- **Market bidding:** The bias could be corrected to improve average performance; the spread (std) determines the risk around any bid.
- **Forecast selection:** When possible, use shorter-horizon forecasts (updated more frequently) for better accuracy.